# Tool Choice

The `tool_choice` parameter controls **how** Claude is allowed to use the tools you provide.
Four modes are available:

| Mode | Behaviour |
|------|-----------| 
| `auto` | Claude decides whether to call a tool — or answer directly — based on the question |
| `tool` | Claude **must** call one specific named tool (great for fixed structured output) |
| `any`  | Claude **must** call one of the provided tools, but chooses which one |
| `none` | Claude **must not** call any tool — responds in plain text even if tools are defined |

This notebook mirrors the Python `tool_choice.ipynb` cookbook, rewritten for TypeScript + the Deno kernel.
Make sure you are comfortable with the basics of tool use before reading this notebook.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL_NAME = 'claude-sonnet-4-6';

console.log('Client ready. Model:', MODEL_NAME);

## `auto` — Claude Decides

Setting `tool_choice` to `{ type: 'auto' }` lets Claude decide whether to call a tool.
This is the **default** behaviour when you pass `tools` without specifying `tool_choice`.

To demonstrate, we define a mock `web_search` tool. Claude will call it for questions
that require up-to-date information and skip it for questions it can answer from training data.

> **Note**: the mock handler just prints a message — no real web request is made.

In [ ]:
// Mock handler — replace with a real search API in production
function webSearch(topic: string): void {
  console.log(`Pretending to search the web for: "${topic}"`);
}

const webSearchTool: Anthropic.Tool = {
  name: 'web_search',
  description:
    'Retrieve up-to-date information on a topic by searching the web. ' +
    'Use this for questions about recent events or facts you cannot confidently answer.',
  input_schema: {
    type: 'object',
    properties: {
      topic: { type: 'string', description: 'The topic to search the web for' },
    },
    required: ['topic'],
  },
};

console.log('Tool defined:', webSearchTool.name);

In [ ]:
async function chatWithWebSearch(userQuery: string): Promise<void> {
  const today = new Date().toLocaleDateString('en-US', {
    year: 'numeric', month: 'long', day: 'numeric',
  });

  const systemPrompt =
    'Answer as many questions as you can using your existing knowledge. ' +
    'Only search the web for queries that you cannot confidently answer. ' +
    `Today's date is ${today}. ` +
    'If a question involves a very recent event, use the search tool.';

  const response = await client.messages.create({
    system: systemPrompt,
    model: MODEL_NAME,
    max_tokens: 1000,
    tool_choice: { type: 'auto' },
    tools: [webSearchTool],
    messages: [{ role: 'user', content: userQuery }],
  });

  const last = response.content[response.content.length - 1];
  if (last.type === 'text') {
    console.log('Claude did NOT call a tool');
    console.log(`Assistant: ${last.text}`);
  } else if (last.type === 'tool_use') {
    console.log('Claude wants to use a tool');
    console.log(`  Tool : ${last.name}`);
    console.log(`  Input: ${JSON.stringify(last.input)}`);
  }
  console.log();
}

// Claude can answer from training data — should NOT call the tool
await chatWithWebSearch('What color is the sky?');

// Requires up-to-date data — SHOULD call the tool
await chatWithWebSearch('Who won the 2024 Miami Grand Prix?');

In [ ]:
// Claude knows this — should NOT call the tool
await chatWithWebSearch('Who won the Super Bowl in 2022?');

// Too recent for training data — SHOULD call the tool
await chatWithWebSearch('Who won the Super Bowl in 2025?');

### Your Prompt Matters

When using `auto`, Claude can be over-eager to call tools without clear guidance.
The system prompt above includes explicit instructions about **when not to call** the tool:

```
Answer as many questions as you can using your existing knowledge.
Only search the web for queries that you cannot confidently answer.
```

Without these lines, Claude may reach for the search tool even for questions it can
answer perfectly well from its training data. Spending time on the system prompt is just
as important as defining the tool schema itself.


## Forcing a Specific Tool

Setting `tool_choice` to `{ type: 'tool', name: '...' }` forces Claude to always call
that exact tool, regardless of the input.

The typical use case is **structured output extraction**: forcing a tool with a strict
`input_schema` guarantees the response shape on every call — no plain text, no schema drift.

We define two tools:

- `print_sentiment_scores` — returns three numeric sentiment scores
- `calculator` — adds two numbers

We first show the `auto` version, where Claude may ignore the sentiment tool or call the
calculator instead. Then we force `print_sentiment_scores` and confirm it is always called.

In [ ]:
const twoTools: Anthropic.Tool[] = [
  {
    name: 'print_sentiment_scores',
    description: 'Prints the sentiment scores of a given piece of text.',
    input_schema: {
      type: 'object',
      properties: {
        positive_score: { type: 'number', description: 'Positive sentiment score, 0.0-1.0' },
        negative_score: { type: 'number', description: 'Negative sentiment score, 0.0-1.0' },
        neutral_score:  { type: 'number', description: 'Neutral sentiment score, 0.0-1.0' },
      },
      required: ['positive_score', 'negative_score', 'neutral_score'],
    },
  },
  {
    name: 'calculator',
    description: 'Adds two numbers together.',
    input_schema: {
      type: 'object',
      properties: {
        num1: { type: 'number', description: 'First number to add' },
        num2: { type: 'number', description: 'Second number to add' },
      },
      required: ['num1', 'num2'],
    },
  },
];

console.log('Tools defined:', twoTools.map(t => t.name));

In [ ]:
// ❌  tool_choice: auto — Claude may give a text reply, or call the calculator instead
async function analyzeTweetAuto(query: string): Promise<void> {
  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 1024,
    tools: twoTools,
    tool_choice: { type: 'auto' },
    messages: [{ role: 'user', content: query }],
  });

  const last = response.content[response.content.length - 1];
  console.log(`stop_reason: ${response.stop_reason}`);
  if (last.type === 'text') {
    console.log('TEXT reply (sentiment tool was NOT called):', last.text.slice(0, 120));
  } else if (last.type === 'tool_use') {
    console.log(`TOOL called: ${last.name}`, JSON.stringify(last.input));
  }
  console.log();
}

// Claude may respond with plain text instead of calling the sentiment tool
await analyzeTweetAuto('Holy cow, I just made the most incredible meal!');

// Math hint in the tweet — Claude may call the calculator instead of sentiment!
await analyzeTweetAuto('I love my cats! I had four and just adopted 2 more! Guess how many I have now?');

In [ ]:
// ✅  tool_choice: { type: 'tool', name: '...' } — always calls print_sentiment_scores
async function analyzeTweetSentiment(query: string): Promise<void> {
  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 1024,
    tools: twoTools,
    tool_choice: { type: 'tool', name: 'print_sentiment_scores' },
    messages: [{ role: 'user', content: query }],
  });

  const block = response.content.find(b => b.type === 'tool_use');
  if (block?.type === 'tool_use') {
    type Scores = { positive_score: number; negative_score: number; neutral_score: number };
    const s = block.input as Scores;
    console.log(`Tweet  : "${query.slice(0, 65)}"`);
    console.log(`Scores : pos=${s.positive_score.toFixed(2)}  neg=${s.negative_score.toFixed(2)}  neu=${s.neutral_score.toFixed(2)}`);
    console.log();
  }
}

// Both calls always produce sentiment scores — never a text reply or calculator result
await analyzeTweetSentiment('Holy cow, I just made the most incredible meal!');
await analyzeTweetSentiment('I love my cats! I had four and just adopted 2 more! Guess how many I have now?');

### Prompt Engineering Still Matters

Forcing a specific tool guarantees the **schema** but not the **quality** of the scores.
A well-structured prompt gives Claude clearer context and yields more accurate results:

```typescript
// Minimal — works, but Claude has little context
messages: [{ role: 'user', content: tweet }]

// Better — explicit instruction + XML wrapper prevents content/instruction confusion
messages: [{ role: 'user', content: `Analyse the sentiment of this tweet:\n<tweet>${tweet}</tweet>` }]
```


## `any` — Must Use a Tool, Claude Picks Which

`tool_choice: { type: 'any' }` tells Claude: *"you must call one of these tools,
but you decide which one."* Claude never responds with plain text.

This is useful when your application can only act through structured tool calls —
for example, an SMS chatbot where every reply must go out as a text message.

We give Claude two tools:

- `send_text_to_user` — the only way the bot can communicate with a user
- `get_customer_info` — looks up order history by username

The system prompt instructs Claude not to call `get_customer_info` until the user
has provided their username. Claude must always call one of the two tools.

In [ ]:
// Mock handlers
function sendTextToUser(text: string): void {
  console.log(`TEXT MESSAGE SENT: ${text}`);
}

function getCustomerInfo(username: string): object {
  return {
    username,
    email: `${username}@email.com`,
    purchases: [
      { id: 1, product: 'computer mouse' },
      { id: 2, product: 'screen protector' },
      { id: 3, product: 'usb charging cable' },
    ],
  };
}

const smsTools: Anthropic.Tool[] = [
  {
    name: 'send_text_to_user',
    description: 'Sends a text message to the user.',
    input_schema: {
      type: 'object',
      properties: {
        text: { type: 'string', description: 'The text message content to send' },
      },
      required: ['text'],
    },
  },
  {
    name: 'get_customer_info',
    description:
      'Gets information on a customer based on their username. ' +
      'Returns email, username, and previous purchases. ' +
      'Only call this tool once the user has provided their username.',
    input_schema: {
      type: 'object',
      properties: {
        username: { type: 'string', description: 'The username of the customer' },
      },
      required: ['username'],
    },
  },
];

const smsSystem =
  'All communication with a user is done via text message. ' +
  'Only call tools when you have enough information to do so accurately. ' +
  'Do not call get_customer_info until the user has provided their username. ' +
  'If you do not know the username, ask for it via send_text_to_user.';

async function smsChatbot(userMessage: string): Promise<void> {
  console.log(`User: "${userMessage}"`);

  const response = await client.messages.create({
    system: smsSystem,
    model: MODEL_NAME,
    max_tokens: 1024,
    tools: smsTools,
    tool_choice: { type: 'any' },
    messages: [{ role: 'user', content: userMessage }],
  });

  if (response.stop_reason === 'tool_use') {
    const block = response.content.find(b => b.type === 'tool_use');
    if (block?.type === 'tool_use') {
      console.log(`=== Claude calls: ${block.name} ===`);
      if (block.name === 'send_text_to_user') {
        sendTextToUser((block.input as { text: string }).text);
      } else if (block.name === 'get_customer_info') {
        console.log(JSON.stringify(getCustomerInfo((block.input as { username: string }).username), null, 2));
      } else {
        console.log('Unknown tool:', block.name);
      }
    }
  } else {
    // With tool_choice: any this branch should never execute
    console.log('No tool called — this should not happen with tool_choice: any');
  }
  console.log();
}

console.log('smsChatbot ready.');

In [ ]:
// 1. Simple greeting — no username yet, Claude sends a welcome text
await smsChatbot('Hey there! How are you?');

// 2. Order lookup without username — Claude must ask for it via text message
await smsChatbot('I need help looking up an order');

// 3. Order lookup WITH username — Claude should call get_customer_info
await smsChatbot('I need help looking up an order. My username is jenny76');

// 4. Gibberish — Claude still must call one of the two tools (tool_choice: any)
await smsChatbot('askdj aksjdh asjkdbhas kjdhas 1+1 ajsdh');

## `none` — No Tool Use Allowed

`tool_choice: { type: 'none' }` prevents Claude from calling any tool, even when tool definitions
are present in the request. Claude always responds in plain text.

This is the **default behaviour** when no `tools` parameter is provided at all.

Useful when:
- You want to **temporarily disable tool use** without restructuring your API call
- You need a plain-text baseline answer from the same client setup that normally uses tools
- **Debugging**: confirming what Claude would say before tools are involved

In [ ]:
// tool_choice: none — Claude responds in plain text even though tools are available
async function analyzeWithNoTools(query: string): Promise<void> {
  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 1024,
    tools: twoTools,                          // tools ARE defined …
    tool_choice: { type: 'none' },            // … but Claude must NOT call any of them
    messages: [{ role: 'user', content: query }],
  });

  const last = response.content[response.content.length - 1];
  console.log(`stop_reason: ${response.stop_reason}`);
  if (last.type === 'text') {
    console.log('TEXT reply (no tool called):', last.text.slice(0, 150));
  } else if (last.type === 'tool_use') {
    console.log('UNEXPECTED tool call:', last.name);
  }
  console.log();
}

// Both prompts would trigger a tool under `auto` — but `none` forces plain-text responses
await analyzeWithNoTools('Holy cow, I just made the most incredible meal!');
await analyzeWithNoTools('What is 3 + 4?');

## Summary

| `tool_choice` value | When to use |
|---------------------|-------------|
| `{ type: 'auto' }` | General assistants — Claude calls a tool only when it needs one. Write a clear system prompt to prevent over-calling. |
| `{ type: 'tool', name: '...' }` | Structured output extraction — guarantees a fixed JSON schema on every call. |
| `{ type: 'any' }` | Applications that must always respond via a tool (e.g. SMS bots, event-driven pipelines). |
| `{ type: 'none' }` | Temporarily disable tool use without removing tool definitions. Default when no `tools` are provided. |

### Key takeaways

- **`auto` + good system prompt** avoids unnecessary tool calls for knowledge Claude already has
- **Forcing a specific tool** is the reliable path for structured extraction — avoid plain JSON prompts in production
- **`any`** guarantees a tool is always called, but tool selection is still left to Claude
- **`none`** blocks all tool calls even if tool definitions are present in the request
- Calling a tool and *executing* it are separate steps: Claude outputs a `tool_use` block; your code runs the function

### What to explore next

- `extracting_structured_json.ipynb` — deep dive into `input_schema` patterns (NER, classification, flexible schemas)
- `tool_selection_basics.ipynb` — the agentic loop and `BoundTool` pattern for multi-turn tool conversations
- `automatic-context-compaction.ipynb` — managing long multi-turn tool conversations